# Add AI-labeled (standardized) data to DuckDB

**Mission**: Ingest the LLM-standardization batch outputs (JSONL) into the bronze layer of `/workspace/ICOCO.duckdb`, one table per `{task folder}.{output version}`.

| DuckDB schema (table name) | Source folder under `/workspace/data` | Output |
|---|---|---|
| `scientific_name.output1/2/3` | `scientific_name` | recognized **scientific names** (list) |
| `odor.output1/2/3` | `odor` | perceived **odors** (list) |
| `cuisine_normalize.output1/2/3` | `cuisine_normalize` | **country / sub-region / continent** (object) |
| `ingredient_standardize.output1/2/3` | `ingredient_standardize` | **normalized ingredient** (list) |

**Principles**
- **Idempotent** - every table is built with `CREATE OR REPLACE`, so the notebook can be re-run safely.
- **Python only for parsing** - extracting the nested answer and converting the JSON array/object into DuckDB `VARCHAR[]` is far more robust in Python (SQL string parsing would be fragile here).
- **One DuckDB connection** - we reuse the single `%sql` connection (DuckDB limitation).

In [1]:
%load_ext sql
%config SqlMagic.displaylimit = 100  # show all rows in query results
%sql duckdb:////workspace/ICOCO.duckdb

Connecting to 'duckdb:////workspace/ICOCO.duckdb'

In [2]:
# Close the connection FIRST to flush the WAL file (ICOCO.duckdb.wal)
# and avoid data corruption from an aborted transaction.
# NOTE: the connection name (URL) is required as an argument.
%sql --close duckdb:////workspace/ICOCO.duckdb

In [3]:
# Reconnect with a fresh connection (recovers/checkpoints the WAL)
%sql duckdb:////workspace/ICOCO.duckdb

Connecting and switching to connection 'duckdb:////workspace/ICOCO.duckdb'

## 0. Create the target schemas (if not exist)

`scientific_name`, `odor`, `cuisine_normalize`, `ingredient_standardize` follow the DuckDB convention that the **table name is the folder of the source data**.

In [4]:
%%sql
CREATE SCHEMA IF NOT EXISTS scientific_name;
CREATE SCHEMA IF NOT EXISTS odor;
CREATE SCHEMA IF NOT EXISTS cuisine_normalize;
CREATE SCHEMA IF NOT EXISTS ingredient_standardize;

Running query in 'duckdb:////workspace/ICOCO.duckdb'

Count


## 1. How the raw LLM outputs are structured

Each line of `outputN.jsonl` is a raw OpenAI-style `/v1/chat/completions` response. The real answer is **deeply nested**: `response.body.choices[0].message.content` holds a **STRING** that is itself JSON. `input_text` is NOT in the response, so it is joined from `merged_results.jsonl` via `custom_id`.

```
{"custom_id": "item-00002", "response": {"status_code": 200, "body": {
    "model": "gemini-2.5-flash-lite",
    "choices": [{"message": {"content": "[\"Vitis vinifera\"]", ...}}],
    "usage": {"promptTokens": 203, "completionTokens": 1}}}}
```

**Content shape differs by task** (unified schema below):
- **list-type** (`scientific_name`, `odor`, `ingredient_standardize`): `content` is a JSON array -> `value_list`.
- **object-type** (`cuisine_normalize`): `content` is a JSON object -> `country_list`, `sub_region_list`, `continent_list`.

**Two troubles discovered while parsing**:
1. Some responses are wrapped in markdown code fences (e.g. `["Vanilla planifolia"]``` `) - must be stripped before `json.loads`.
2. Empty lists are kept as real empty arrays `[]` (not `NULL`); `NULL` is reserved for *not applicable* fields (e.g. `country_list` on a list-type row) or for rows whose content failed to parse (`parsed = FALSE`).

In [5]:
import json
import os
import pandas as pd
from sql.connection import ConnectionManager

DATA_DIR = '/workspace/data'

# Task folders -> target DuckDB schema (schema name == source folder name)
DATASETS = ['scientific_name', 'odor', 'cuisine_normalize', 'ingredient_standardize']
VERSIONS = ['output1', 'output2', 'output3']  # N = LLM answer replicas


def strip_markdown_fence(raw):
    # Some responses come wrapped in ``` code fences -> strip before json.loads.
    if raw is None:
        return None
    s = raw.strip()
    if s.startswith('```'):
        s = s[3:].strip()
    if s.endswith('```'):
        s = s[:-3].strip()
    return s


def load_input_map(folder):
    # custom_id -> original input text (merged_results.jsonl already pairs them).
    m = {}
    path = os.path.join(DATA_DIR, folder, 'merged_results.jsonl')
    with open(path, encoding='utf-8') as f:
        for line in f:
            o = json.loads(line)
            m[o['custom_id']] = o.get('input')
    return m


def parse_output_file(folder, version, input_map):
    # Flatten one output*.jsonl into a list of dicts (one per custom_id).
    records = []
    path = os.path.join(DATA_DIR, folder, f'{version}.jsonl')
    with open(path, encoding='utf-8') as f:
        for line in f:
            o = json.loads(line)
            resp = o.get('response', {}) or {}
            body = resp.get('body', {}) or {}
            choices = body.get('choices') or []
            content = (choices[0].get('message') or {}).get('content') if choices else None
            usage = body.get('usage') or {}

            status = resp.get('status_code')
            cleaned = strip_markdown_fence(content)
            parsed = False
            value_list = None
            country = sub_region = continent = None
            if status == 200 and cleaned is not None:
                try:
                    data = json.loads(cleaned)
                    parsed = True
                    if isinstance(data, list):
                        value_list = data          # keep [] as empty array
                    elif isinstance(data, dict):
                        country = data.get('country_list') or []
                        sub_region = data.get('sub_region_list') or []
                        continent = data.get('continent_list') or []
                except Exception:
                    parsed = False

            records.append({
                'custom_id': o.get('custom_id'),
                'input_text': input_map.get(o.get('custom_id')),
                'status_code': status,
                'model': body.get('model'),
                'parsed': parsed,
                'value_list': value_list,
                'country_list': country,
                'sub_region_list': sub_region,
                'continent_list': continent,
                'content_raw': content,
                'prompt_tokens': usage.get('promptTokens'),
                'completion_tokens': usage.get('completionTokens'),
            })
    return records


# Reuse the SINGLE DuckDB connection opened by %sql (only one connection at a time).
conn = ConnectionManager().current.connection_sqlalchemy
con = conn.connection.driver_connection

loaded = {}
for folder in DATASETS:
    input_map = load_input_map(folder)
    for version in VERSIONS:
        df = pd.DataFrame(parse_output_file(folder, version, input_map))
        # token counts must stay INTEGER (nullable) so DuckDB stores them as BIGINT,
        # not FLOAT (pandas upcasts a numeric column to float as soon as one value is missing).
        df['prompt_tokens'] = df['prompt_tokens'].astype('Int64')
        df['completion_tokens'] = df['completion_tokens'].astype('Int64')
        con.register('_stage', df)
        con.execute(f'CREATE OR REPLACE TABLE "{folder}"."{version}" AS SELECT * FROM _stage')
        con.unregister('_stage')
        loaded[f'{folder}.{version}'] = len(df)

print('Loaded row counts:')
for k, v in loaded.items():
    print(f'  {k}: {v}')

Loaded row counts:
  scientific_name.output1: 4280
  scientific_name.output2: 4280
  scientific_name.output3: 4280
  odor.output1: 3792
  odor.output2: 3792
  odor.output3: 3792
  cuisine_normalize.output1: 116
  cuisine_normalize.output2: 116
  cuisine_normalize.output3: 116
  ingredient_standardize.output1: 17017
  ingredient_standardize.output2: 17017
  ingredient_standardize.output3: 17017


## 2. Verification

Expect every table to contain the same number of rows as lines in its source `outputN.jsonl`, and rare `parsed = FALSE` rows (failed JSON parse) to be kept as-is rather than dropped.

In [6]:
%%sql
SELECT 'scientific_name.output1' tbl, COUNT(*) row_count FROM scientific_name.output1
UNION ALL SELECT 'scientific_name.output2', COUNT(*) FROM scientific_name.output2
UNION ALL SELECT 'scientific_name.output3', COUNT(*) FROM scientific_name.output3
UNION ALL SELECT 'odor.output1', COUNT(*) FROM odor.output1
UNION ALL SELECT 'odor.output2', COUNT(*) FROM odor.output2
UNION ALL SELECT 'odor.output3', COUNT(*) FROM odor.output3
UNION ALL SELECT 'cuisine_normalize.output1', COUNT(*) FROM cuisine_normalize.output1
UNION ALL SELECT 'cuisine_normalize.output2', COUNT(*) FROM cuisine_normalize.output2
UNION ALL SELECT 'cuisine_normalize.output3', COUNT(*) FROM cuisine_normalize.output3
UNION ALL SELECT 'ingredient_standardize.output1', COUNT(*) FROM ingredient_standardize.output1
UNION ALL SELECT 'ingredient_standardize.output2', COUNT(*) FROM ingredient_standardize.output2
UNION ALL SELECT 'ingredient_standardize.output3', COUNT(*) FROM ingredient_standardize.output3
ORDER BY tbl;

Running query in 'duckdb:////workspace/ICOCO.duckdb'

tbl,row_count
cuisine_normalize.output1,116
cuisine_normalize.output2,116
cuisine_normalize.output3,116
ingredient_standardize.output1,17017
ingredient_standardize.output2,17017
ingredient_standardize.output3,17017
odor.output1,3792
odor.output2,3792
odor.output3,3792
scientific_name.output1,4280


In [7]:
%%sql
-- two list-type samples + one object-type sample
SELECT * FROM scientific_name.output1 LIMIT 2;
SELECT * FROM cuisine_normalize.output1 LIMIT 2;

Running query in 'duckdb:////workspace/ICOCO.duckdb'

custom_id,input_text,status_code,model,parsed,value_list,country_list,sub_region_list,continent_list,content_raw,prompt_tokens,completion_tokens
item-00001,Ethiopian,200,gemini-2.5-flash-lite,True,None,['Ethiopia'],['Eastern Africa'],['Africa'],"{""country_list"": [""Ethiopia""], ""sub_region_list"": [""Eastern Africa""], ""continent_list"": [""Africa""]}",259,28
item-00002,British,200,gemini-2.5-flash-lite,True,None,['United Kingdom'],['Northern Europe'],['Europe'],"{""country_list"": [""United Kingdom""], ""sub_region_list"": [""Northern Europe""], ""continent_list"": [""Europe""]}",258,28


In [8]:
# Close ONLY when done - flush WAL to disk and release the DB.
%sql --close duckdb:////workspace/ICOCO.duckdb